In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata
from sklearn.preprocessing import MinMaxScaler

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/higashi'

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/embedding/higashi_filtered.h5ad')

In [ ]:
adata

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/xgboost_time_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
traj_score_path = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster/csv/gene_score/traj_score.csv'
traj_score = pd.read_csv(traj_score_path, sep='\t', index_col=0)
traj_score.shape

In [ ]:
traj_score.head()

In [ ]:
adata.obs['traj_score'] = traj_score.reindex(adata.obs.index)['traj_score']
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['cluster']
adata.obs['cluster_sub'] = clusters.reindex(adata.obs.index)['cluster_sub']
tmp_adata = adata[~adata.obs['cluster'].isna()]
tmp_adata

In [ ]:
sc.pl.umap(adata, color=['line', 'time'])

### use integrate_m3c now

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(clusters_path, sep='\t', index_col=0)
leiden['leiden'] = [f'{x}_{y}' for x,y in zip(leiden['line'], leiden['final_leiden'])]
leiden.shape

In [ ]:
leiden.head()

In [ ]:
adata.obs['final_leiden'] = leiden.reindex(adata.obs.index)['final_leiden']
tmp_adata = adata[~adata.obs['final_leiden'].isna()]
tmp_adata.shape

In [ ]:
umap_df = adata.obs[['line', 'time', 'cluster', 'final_leiden']].copy()
umap_df.columns = ['line', 'time', 'cluster', 'mc_leiden']
umap_df['umap0'] = adata.obsm['X_umap'][:,0]
umap_df['umap1'] = adata.obsm['X_umap'][:,1]

select = ['time', 'traj_score', 'cluster']
for s in select:
    umap_df[s] = adata.obs[s]

In [ ]:
umap_df.head()

In [ ]:
scaler = MinMaxScaler()
umap_df['norm_traj'] = np.ravel(scaler.fit_transform(umap_df['traj_score'].values.reshape(-1,1)))

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
umap_df.to_csv(f'{indir}/snm3c_higashi_umap.csv', sep='\t')

In [ ]:
!date